# Lab 8: Building and Improving a Document RAG System

In [1]:
import os
from openai import OpenAI

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from IPython.display import display, Markdown

client = OpenAI()

Throughout this unit, you have explored how RAG systems work and built the individual pieces: embeddings and vector similarity, document loaders, chunking strategies, vector indexes, RAG chains, and query transformation techniques. In this lab, you will assemble those pieces into a complete RAG application for an outdoor gear company, then improve its retrieval quality by adding a query rewriting step on top.

You will complete the following tasks:

1. Load and chunk the knowledge base:
    * Load a customer support corpus from a single text file
    * Choose chunking parameters and justify your choice in writing
2. Build the vector index:
    * Initialize an embedding model and a Chroma vector store
    * Inspect retrieval quality on a test query using similarity scores
3. Write the RAG instruction prompt:
    * Write a prompt template that grounds the model in retrieved context and handles unknowns
4. Assemble the baseline RAG chain:
    * Connect the retriever, prompt, LLM, and output parser using the pipe operator
    * Test the chain on three different kinds of customer queries
5. Add query rewriting and compare:
    * Write a query rewriting prompt that expands short customer queries
    * Compare baseline answers to rewritten-query answers on the same three queries
6. Analysis:
    * Evaluate the pipeline's behavior, limitations, and business implications
7. AI Reflection:
    * Reflect on how you used (or did not use) AI tools during this lab

<div style="border:1px solid #ccc; border-radius:8px; padding:12px; background-color:#f8f9fa;">

<p><strong>Important:</strong> All graded cells that require you to enter code will contain regions marked with <code># YOUR CODE HERE</code> and <code># END OF YOUR CODE</code>. Between these lines, you’ll find the line <code>raise NotImplementedError("Your code is missing.")</code>, like so:</p>

```python
# YOUR CODE HERE
raise NotImplementedError("Your code is missing.")
# END OF YOUR CODE
```
<p></p>
<p>These markers indicate exactly where you should enter your answer. Replace the line <code>raise NotImplementedError("Your code is missing.")</code> with your code.

## Business Context

Read through the scenario below. You will be putting yourself in the shoes of a junior ML engineer (MLE) at Marlowe & Finch, where you have been asked to prototype a customer support assistant for the company's website.

#### 1. Company and Context

Marlowe & Finch is a small outdoor gear company based in Boulder, Colorado. The company was founded in 2017 by two former trail crew leads who got tired of gear that looked great in the store and fell apart on the trail. Today Marlowe & Finch designs lightweight, three-season backpacking gear for weekend backpackers and thru-hikers, and sells through its own website and a handful of independent outdoor retailers.

The customer base is mostly enthusiastic but not expert. Many customers are buying their first real piece of backcountry gear and have a lot of practical questions before they pull the trigger on a $400 tent.

#### 2. Business Challenge

Marlowe & Finch's customer support team currently answers most questions by hand, working from an internal knowledge base of product specs, warranty terms, and returns/shipping policies. Response times have been slipping as the volume of presale questions grows, and customers who do not get a quick answer often abandon their carts. The team has been asked to build a prototype assistant that can answer common product and policy questions on the website, using the same knowledge base the human agents reference today.

#### 3. Business Goal

The goal is to build a working RAG (Retrieval-Augmented Generation) prototype that can take a customer's question, retrieve the most relevant passages from the Marlowe & Finch knowledge base, and produce a grounded answer. If the prototype performs well, it would be deployed as a first-line assistant on the website. Customer support agents would handle anything the assistant cannot.

#### 4. Your Role and Task

You have just joined Marlowe & Finch as a junior MLE on the Customer Experience team. The team has handed you the customer-facing knowledge base as a single text file and asked you to build and test a prototype RAG assistant. Your job is to make the chunking, retrieval, and prompting decisions, then evaluate whether the prototype produces answers the customer support team would be comfortable putting in front of customers.

This work requires judgment at every stage. The chunking parameters you pick determine what the retriever can find. The retrieval setup determines what context the model sees. The instruction prompt determines whether the model stays grounded in the retrieved context or wanders into territory the knowledge base does not cover. A small change in any of these stages can meaningfully change the answers customers receive.

#### 5. Technical Focus in This Lab

This lab focuses on building and improving a document RAG pipeline:

* **Document Loading and Chunking** &mdash; Loading a source document and splitting it into focused chunks with `RecursiveCharacterTextSplitter`.
* **Vector Indexing and Retrieval** &mdash; Embedding chunks with OpenAI embeddings and storing them in a Chroma vector store for similarity search.
* **Prompt Engineering for RAG** &mdash; Writing an instruction prompt that grounds the model in retrieved context and handles questions the knowledge base cannot answer.
* **Chain Composition with LangChain** &mdash; Using the pipe operator to assemble a complete RAG chain.
* **Query Transformation** &mdash; Adding a query rewriting step in front of the retriever to improve retrieval on short, vague queries.

## Part 1. Load and Chunk the Knowledge Base

Marlowe & Finch has provided you with a single text file containing the customer-facing knowledge base. It is stored at `data/marlowe_knowledge_base.txt` and includes the company's "About" content, specs for three products, warranty terms, returns and shipping policies, and an FAQ.

Before you can search this knowledge base with embeddings, you need to load it and split it into chunks. You practiced both steps in the LangChain primer and the chunking strategies activity.

**Task**: Use `TextLoader` to load the file at `data/marlowe_knowledge_base.txt`. Save the result to a variable called `documents`. Then print how many documents you loaded and the total character length, so you can sanity-check that the file was read correctly.

*Tip*: `TextLoader(path).load()` returns a list of `Document` objects.

In [5]:
# YOUR CODE HERE
documents = TextLoader("data/marlowe_knowledge_base.txt").load()
# END OF YOUR CODE

**Task**: Print the first 1,500 characters of the document so you can see what kind of content you are working with. You will use this view to inform your chunking choices in the next step.

*Tip*: You can slice a string with `text[:1500]`. The full document content lives in `documents[0].page_content`.

In [6]:
# YOUR CODE HERE
documents[0].page_content[:1500]
# END OF YOUR CODE

'==========================================================\nMARLOWE & FINCH CUSTOMER SUPPORT KNOWLEDGE BASE\nLast updated: Spring 2025\n==========================================================\n\nWelcome to the Marlowe & Finch customer support knowledge base. This document\nis the internal source of truth our support team uses to answer customer\nquestions. It covers our current product lineup, warranty terms, returns and\nshipping policies, and frequently asked questions.\n\n==========================================================\nABOUT MARLOWE & FINCH\n==========================================================\n\nMarlowe & Finch is a small outdoor gear company based in Boulder, Colorado.\nWe design backcountry equipment for weekend backpackers and thru-hikers, with\na focus on lightweight three-season gear that holds up to real use. Our\nproducts are sold through our website and a handful of independent outdoor\nretailers. We do not currently sell through Amazon, REI, or other 

Now that you have seen the content, take a moment to think about its structure. It mixes brand story, product specs, policy language, and FAQ entries. A chunk that mixes a product spec with an unrelated policy paragraph would be a noisy hit for either kind of query. A chunk that is too small might cut a policy sentence in half so that neither chunk contains a complete answer.

You will now split the document into chunks. The chunking strategies activity covered the trade-offs between small, medium, and large chunks, and the role of overlap.

**Task**: Create a `RecursiveCharacterTextSplitter` with `chunk_size` and `chunk_overlap` values of your choosing. Then call `.split_documents()` on `documents` to produce the chunks. Save the result to a variable called `chunks`.

After running your splitter, print the total number of chunks and the average chunk length so you can sanity-check your choice.

In [8]:
# YOUR CODE HERE
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75)
chunks = splitter.split_documents(documents)

total_chunks = len(chunks)
avg_chunk_length = sum(len(chunk.page_content) for chunk in chunks) / total_chunks
print(f"Total number of chunks: {total_chunks}")
print(f"Average chunk length: {avg_chunk_length:.2f} characters")
# END OF YOUR CODE

Total number of chunks: 35
Average chunk length: 358.63 characters


**Task**: In the markdown cell below, answer the following:

1. What `chunk_size` and `chunk_overlap` did you choose?
2. Why did you pick those values for this particular knowledge base? Refer to something specific you saw in `documents[0].page_content`.
3. What is one risk of your choice (for example, what kind of customer question might it handle poorly)?

1. I chose a `chunk_size` of 500 characters and a `chunk_overlap` of 150 characters.

2. I picked these values because this knowledge base contains different types of customer support information, including company background, product specifications, warranty terms, and shipping/return policies. From `documents[0].page_content`, I noticed that the document is organized into separate sections such as "ABOUT MARLOWE & FINCH" and "PRODUCT CATALOG." A chunk size of 500 keeps retrieved passages focused on a specific topic, such as a product or policy section, while the 150-character overlap helps preserve important context when information is split between chunks. This is useful for customer questions where the assistant needs to retrieve precise details rather than large portions of the document.

3. One risk of this choice is that smaller chunks may separate related information across multiple chunks. For example, a customer asking "Does the Trailhead 2 Tent come with a warranty and can I return it if it does not fit my needs?" may require information from both the product section and the returns/warranty sections. The retriever may not always retrieve all necessary context, which could lead to an incomplete answer.

## Part 2. Build the Vector Index

Now that you have chunks, you will embed them and store them in a vector database so the retriever can find the most relevant chunks for any customer query. You did this same pipeline in the Build a Vector Index activity.

**Task**: Initialize an `OpenAIEmbeddings` object using the `text-embedding-3-small` model. Save it to a variable called `embeddings`. Then build a Chroma vector store from your chunks using `Chroma.from_documents()`. Save the result to a variable called `vectorstore`.

In [9]:
# YOUR CODE HERE
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents=chunks,
    embedding=embeddings)
# END OF YOUR CODE

**Task**: Create a retriever from your vector store using `vectorstore.as_retriever()`. Configure it to use similarity search and to return the top 4 chunks. Save it to a variable called `retriever`.

*Tip*: `as_retriever()` accepts `search_type` and `search_kwargs` arguments. The `k` value goes inside `search_kwargs`.

In [11]:
# YOUR CODE HERE
retriever = vectorstore.as_retriever(search_type="similarity",
    search_kwargs={"k": 4})
# END OF YOUR CODE

Before you build the full RAG chain, let's inspect what the retriever returns for a real customer query, along with the similarity scores. The cell below uses `similarity_search_with_score()` so you can see how close each retrieved chunk actually is to the query.

In [14]:
# YOUR CODE HERE
query = "What are the warranty terms and coverage for Marlowe & Finch products?"
results = vectorstore.similarity_search_with_score(query)
for i, (doc, score) in enumerate(results):
    print(f"Chunk {i+1}")
    print(f"Similarity score: {score}")
    print(doc.page_content[:500])
    print("-" * 80)
# END OF YOUR CODE

Chunk 1
Similarity score: 0.5610854625701904
All Marlowe & Finch gear is covered by our Trail-Tested Warranty. The
Trail-Tested Warranty covers manufacturing defects and workmanship issues
for the lifetime of the original owner.

What is covered:
- Seam failures
- Zipper failures (excluding damage from forcing a stuck zipper)
- Buckle and strap hardware failures
- Pole breakage during normal use
- Delamination of waterproof coatings within the first three years of
  ownership
- Down or synthetic fill loss through stitching
--------------------------------------------------------------------------------
Chunk 2
Similarity score: 0.6270182132720947
How to file a warranty claim:
1. Email warranty@marloweandfinch.com with your order number, photos of the
   issue, and a brief description of how the issue occurred.
2. Our warranty team reviews claims within 5 business days.
3. If approved, we will either repair the product, replace it, or issue
   store credit, at our discretion.

The warra

**Task**: Look at the output above and think about how well the retriever is working. In the markdown cell below, answer the following:

1. Are the top retrieved chunks actually about the return policy, or are some of them off-topic?
2. What do the distance scores tell you? Is there a clear gap between the top result and the bottom result, or are they similar?
3. If you saw an off-topic chunk in the top 4, what do you think pulled it in? If all four chunks were on-topic, would your answer change if a customer asked something more specific, like "can I return a tent I used once"?

1. The top retrieved chunks are mostly relevant to the warranty policy, but some chunks are off-topic. Chunks 1 and 2 directly discuss the Trail-Tested Warranty, including coverage details and how to file a claim. However, chunks 3 and 4 are not useful for answering a warranty question. Chunk 3 is a general introduction to the knowledge base, and chunk 4 only describes the company background.

2. The distance scores show that the first two chunks are significantly more relevant than the last chunk. The top result has a score of 0.5611, while the fourth result has a score of 0.8639. Since the scores increase as the chunks become less similar, there is a noticeable gap between the most relevant chunks and the off-topic chunks. This suggests that the retriever successfully found the correct warranty information but also included some less relevant context.

3. The off-topic chunks were likely retrieved because they contain general terms related to Marlowe & Finch and its products, which overlap with the customer's query. The embedding model may recognize the company name and product context even though those chunks do not contain the specific warranty information. If the customer asked a more specific question like "Can I return a tent I used once?", I would expect different chunks to be retrieved because the query would contain more specific return-related terms. However, the retriever could still struggle if the return policy uses different wording or is separated from similar product-related information.

## Part 3. Write the RAG Instruction Prompt

You have a retriever. Now you need an instruction prompt that tells the LLM what to do with the retrieved context. You wrote one of these in the Build Your First RAG System activity. In that activity, the prompt was for a generic question-answering assistant. Here you will write one tailored to Marlowe & Finch's customer support tone and to the specific risks of running a customer-facing assistant.

A good RAG prompt for this use case should:

1. Position the model as a customer support assistant for Marlowe & Finch
2. Tell the model to answer using only the retrieved context
3. Tell the model exactly what to say when the context does not contain the answer (for example, that it cannot find that information and the customer should contact customer support at `support@marloweandfinch.com`)
4. Set constraints on tone and length so the answers sound like the support team

**Task**: Create a variable called `rag_instruction` and assign it a prompt template string. The template must include the exact placeholders `{context}` and `{question}` so it can be used with `PromptTemplate.from_template()` later. Write the instruction text yourself, following the four requirements above.

*Tip*: Use triple quotation marks (`"""`) to write a multi-line string.

In [15]:
# YOUR CODE HERE
rag_instruction = """
You are a customer support assistant for Marlowe & Finch, an outdoor gear company specializing in lightweight backpacking equipment.

Answer customer questions using only the information provided in the retrieved context below. Do not use outside knowledge or make assumptions. If the retrieved context does not contain enough information to answer the customer's question, politely explain that you cannot find that information and ask the customer to contact Marlowe & Finch customer support at support@marloweandfinch.com.

Keep responses clear, helpful, and professional. Use a friendly customer support tone. Keep answers concise while including important details such as policies, product specifications, or next steps when available.

Retrieved context:
{context}

Customer question:
{question}

Answer:
"""
# END OF YOUR CODE

The cell below builds a `PromptTemplate` object from the string you wrote and prints the input variables it detected. You should see `['context', 'question']`.

In [16]:
# Do not remove or edit this cell

prompt = PromptTemplate.from_template(rag_instruction)
print("PromptTemplate input variables:", prompt.input_variables)

PromptTemplate input variables: ['context', 'question']


## Part 4. Assemble the Baseline RAG Chain

Now you will assemble the complete RAG chain that you built in the Build Your First RAG System activity. The components are the same; only the corpus, the prompt, and the queries are different.

First, initialize the LLM.

In [17]:
# Do not remove or edit this cell

llm = ChatOpenAI(model="gpt-4o")

**Task**: Write a function called `format_docs` that takes a list of `Document` objects and returns a single string containing their `page_content`, separated by two newlines (`\n\n`).

In [18]:
# YOUR CODE HERE
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
# END OF YOUR CODE

**Task**: Assemble the baseline RAG chain. Save it to a variable called `rag_chain`. The chain should use:
- A dictionary mapping where `"context"` runs the retriever and pipes its output through `format_docs`, and `"question"` uses `RunnablePassthrough()`
- The `prompt` object created in Part 3
- The `llm` object initialized above
- A `StrOutputParser()` at the end

Use the pipe operator (`|`) to connect the components.

In [20]:
# YOUR CODE HERE
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
# END OF YOUR CODEa

Now let's test the chain on three customer queries. These are written to feel like what real shoppers actually type into a chat box. Pay attention to how the chain handles each one. They are deliberately different from each other:

- **Query 1** is short and underspecified, the kind of thing someone types when they have a quick question in their head.
- **Query 2** is a specific, scenario-based question about a real Marlowe & Finch policy.
- **Query 3** is a specific question that tests a particular edge case in Marlowe & Finch's policies.

In [21]:
# Do not remove or edit this cell

baseline_queries = [
    "is the tent waterproof",
    "can i return a tent i used on a weekend trip",
    "do you ship to australia"
]

baseline_answers = {}

for q in baseline_queries:
    answer = rag_chain.invoke(q)
    baseline_answers[q] = answer
    print(f"Q: {q}")
    print(f"A: {answer}\n")
    print("-" * 60)

Q: is the tent waterproof
A: The Trailhead 2 tent has a 2,000 mm hydrostatic head rating on both the fly and the floor, with fully taped seams, making it suitable for steady rain and moderate wind. However, it is not rated for sustained heavy rain, alpine storms, or snow loading. If you need a tent for those conditions, consider the Summit 4 four-season tent.

------------------------------------------------------------
Q: can i return a tent i used on a weekend trip
A: Thank you for reaching out! Based on our return policy, items that have been used outdoors, including tents, are not eligible for return, even if they appear clean. However, if your tent has a manufacturing defect, it may be covered under our Trail-Tested Warranty. For more assistance or to discuss warranty coverage, please contact Marlowe & Finch customer support at support@marloweandfinch.com.

------------------------------------------------------------
Q: do you ship to australia
A: I'm sorry, but it looks like we c

**Task**: Review the three answers above. In the markdown cell below, answer the following:

1. How did the chain handle each of the three queries? Be specific: which ones produced clean answers, and which ones had problems? Were there any queries that the chain should have been able to answer but did not answer correctly? 
2. If a customer support manager saw these three answers, which one would they be most uncomfortable publishing on the website, and why?

1. The chain handled all three queries well overall. The first query about waterproofing produced a clean answer that correctly explained the Trailhead 2's waterproof rating and limitations. The second query about returning a used tent also produced a strong answer because it correctly applied the return policy and mentioned warranty coverage as an alternative.

The third query about shipping to Australia appears to produce a clean answer as well, assuming the shipping policy information was included in the retrieved context. The response is concise and provides a support contact for additional help. However, if the knowledge base did not contain information about international shipping, this would be a problem because the model would be making an unsupported claim.

2. The answer a customer support manager would be most concerned about would depend on whether the shipping information exists in the knowledge base. If it does not, the Australia shipping response would be the biggest concern because inaccurate shipping information could directly affect customer purchases. Otherwise, all three answers are appropriately grounded and customer-ready.

## Part 5. Add Query Rewriting and Compare

Short, casual queries are a known weakness of vector retrieval. A query like "is the tent waterproof" is short and underspecified compared to the full-sentence documentation in the knowledge base.

In the query transformation activity, you saw that one way to address this is to rewrite the query into a longer, more complete version before sending it to the retriever. In this part, you will add a query rewriting step in front of the baseline chain and compare the results.

**Task**: Create a variable named `rewrite_prompt_template` that contains a prompt for rewriting a customer query. This prompt will be combined with the actual customer query using `.format()` in the next cell. Use the placeholder `{short_query}` where the customer's original query should appear.

Your prompt should:

1. Explain the task (rewrite a short customer query into a more complete version)
2. Make it clear that the rewritten query should preserve the customer's intent
3. Ask for exactly one rewritten query as output, not a list or numbered alternatives

*Note*: Because this template contains a curly-brace placeholder, define it as a regular string now and use `.format()` later, rather than as an f-string directly.

*Tip*: This is similar to the query rewriting work you did in the Improve RAG Retrieval Through Query Transformation activity, but there you asked for multiple alternatives. Here you want exactly one rewritten query so you can drop it straight into the existing retriever.

In [22]:
# YOUR CODE HERE
rewrite_prompt_template = """
Rewrite the following short customer query into a more complete and specific search query that will help retrieve relevant information from the Marlowe & Finch customer support knowledge base.

Preserve the customer's original intent. Add useful context and clarify the request, but do not change what the customer is asking. Output exactly one rewritten query. Do not provide a list, explanations, or numbered alternatives.

Customer query:
{short_query}

Rewritten query:
"""
# END OF YOUR CODE

The cell below uses your `rewrite_prompt_template` to rewrite each of the three queries from Part 4. To keep things fast and consistent, we rewrite each query once and save the results in a dictionary called `rewritten_queries`. That way, when we compare baseline answers to rewritten-query answers, we are using the same rewrites throughout.

In [23]:
# Do not remove or edit this cell

rewritten_queries = {}

for q in baseline_queries:
    filled_prompt = rewrite_prompt_template.format(short_query=q)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": filled_prompt}]
    )
    rewritten_queries[q] = response.choices[0].message.content.strip()
    print(f"Original:  {q}")
    print(f"Rewritten: {rewritten_queries[q]}\n")

Original:  is the tent waterproof
Rewritten: Is the tent waterproof, and are there specific models offered by Marlowe & Finch that have waterproof features, including details about the level of water resistance and any additional protective treatments applied to the fabric?

Original:  can i return a tent i used on a weekend trip
Rewritten: Can I return a tent that I purchased from Marlowe & Finch after using it on a weekend trip?

Original:  do you ship to australia
Rewritten: Does Marlowe & Finch offer international shipping to Australia, and what are the associated costs and delivery times?



Now let's run the full pipeline (the rewritten query through the existing RAG chain) and compare the answers side by side with the baseline.

In [24]:
# Do not remove or edit this cell

for q in baseline_queries:
    rewritten = rewritten_queries[q]
    rewritten_answer = rag_chain.invoke(rewritten)

    print(f"Original query: {q}")
    print(f"Rewritten query: {rewritten}\n")
    display(Markdown(f"**Baseline answer:** {baseline_answers[q]}"))
    display(Markdown(f"**Rewritten-query answer:** {rewritten_answer}"))
    print("=" * 60)

Original query: is the tent waterproof
Rewritten query: Is the tent waterproof, and are there specific models offered by Marlowe & Finch that have waterproof features, including details about the level of water resistance and any additional protective treatments applied to the fabric?



**Baseline answer:** The Trailhead 2 tent has a 2,000 mm hydrostatic head rating on both the fly and the floor, with fully taped seams, making it suitable for steady rain and moderate wind. However, it is not rated for sustained heavy rain, alpine storms, or snow loading. If you need a tent for those conditions, consider the Summit 4 four-season tent.

**Rewritten-query answer:** The Trailhead 2 tent from Marlowe & Finch offers a level of water resistance with a 2,000 mm hydrostatic head rating on both the fly and the floor, and it has fully taped seams. This tent performs well in steady rain and moderate wind but is not rated for sustained heavy rain, alpine storms, or snow loading. If you're looking for a tent suitable for those harsher conditions, you might consider the Summit 4 four-season tent. If you have more specific questions or need further assistance, please contact Marlowe & Finch customer support at support@marloweandfinch.com.

Original query: can i return a tent i used on a weekend trip
Rewritten query: Can I return a tent that I purchased from Marlowe & Finch after using it on a weekend trip?



**Baseline answer:** Thank you for reaching out! Based on our return policy, items that have been used outdoors, including tents, are not eligible for return, even if they appear clean. However, if your tent has a manufacturing defect, it may be covered under our Trail-Tested Warranty. For more assistance or to discuss warranty coverage, please contact Marlowe & Finch customer support at support@marloweandfinch.com.

**Rewritten-query answer:** Thank you for reaching out. According to our return policy, items that have been used outdoors are not eligible for return, even if they appear clean. However, if your tent has a manufacturing defect, it may be covered under our Trail-Tested Warranty. For warranty claims, please email warranty@marloweandfinch.com with your order number, photos of the issue, and a brief description of how the issue occurred.

If you have any more questions or need further assistance, feel free to contact our customer support at support@marloweandfinch.com.

Original query: do you ship to australia
Rewritten query: Does Marlowe & Finch offer international shipping to Australia, and what are the associated costs and delivery times?



**Baseline answer:** I'm sorry, but it looks like we currently do not ship to Australia. For more detailed information or assistance, please contact Marlowe & Finch customer support at support@marloweandfinch.com.

**Rewritten-query answer:** Thank you for reaching out with your question! Based on the information we have, Marlowe & Finch ships internationally to Canada, the United Kingdom, and most EU countries. Unfortunately, Australia is not listed among the countries we currently ship to. For further assistance or to check if there have been any recent changes, I recommend contacting Marlowe & Finch customer support at support@marloweandfinch.com. They will be happy to help!

**Task**: Review the side-by-side comparison above. In the markdown cell below, answer the following:

1. On which of the three queries did query rewriting visibly change the answer? Did it improve the answer, make it worse, or leave it about the same?
2. Look at the rewritten queries themselves. Are they faithful to the original intent, or did the rewriting step change what the customer was asking about?
3. Query rewriting adds an extra LLM call before every retrieval, which costs both money and a bit of latency. Based on what you saw, is the improvement worth the added cost for this use case? When would you turn it on, and when would you leave it off?

1. For the waterproof tent qustion, the rewritten-query answer was very similar to the baseline answer, only adding a slightly more detailed explanation and a support contact. For the return policy question, the rewritten answer was also similar but slightly improved because it included the specific warranty claim process. For the shipping question, the rewritten query produced the biggest change by adding more specific international shipping details, making the answer more informative than the baseline.

2. Some rewrites added extra details that the customer did not explicitly ask about, such as asking about "specific models" and "additional protective treatments" for the waterproofing question, or asking about shipping costs and delivery times for Australia. These additions could slightly change the scope of the customer's original request.

3. For specific questions that already contain important keywords, such as warranty or return questions, the baseline retriever already performs well, so the extra LLM call may not provide enough benefit to justify the added cost and latency. I would enable query rewriting for short, vague, or underspecified questions like "is it good?" or "does it work?" where additional context can help retrieval. I would leave it off for detailed queries that already clearly describe the customer's intent.

## Part 6. Analysis

You have now built a complete RAG assistant for Marlowe & Finch and tested a query rewriting improvement on top of it. In this section, reflect on the system as a whole.

Answer the following questions in the markdown cell below:

1. **Explaining the system to the team**: Marlowe & Finch's customer support manager is not technical. They want to know, in plain language, why the assistant sometimes gives a great answer and sometimes gives a vague one. Using what you observed in this lab, write a short explanation (3-5 sentences) you could give the manager. Avoid complicated technical jargon. Do not assume they know what an embedding is.

2. **Biggest deployment risk**: Marlowe & Finch is considering putting this assistant live on the website as a first-line responder to customer questions. Based on what you observed in this lab, what is the single biggest risk of deploying this prototype as-is? Name one specific type of customer question that would likely cause the assistant to behave badly, and explain what you would recommend to the team before going live.

1. The assistant gives great answers when it can find a clear match between the customer's question and the information in the company's knowledge base. It may give vague or less reliable answers when the customer's question is too short, unclear, or uses different wording than the documentation. In those cases, the assistant may find information that is related but not exactly what the customer needs. Before using it with customers, we should continue testing common customer questions and improve how the assistant finds and uses the right information.

2. The biggest deployment risk is that the assistant may provide incorrect or unsupported information instead of admitting that it does not know the answer. For example, a customer asking about a specific shipping exception or a policy that is not clearly documented could receive a confident but inaccurate response. Before going live, I would recommend evaluating the assistant on a larger set of real customer questions and having human support available for unclear requests.

## Part 7. Reflection: AI Usage

1. Did you use AI tools for this lab? If yes, which ones and at what points in your work? If no, briefly explain your reasoning.
2. If you used AI, describe one specific prompt that was useful and explain why it worked. If you did not use AI, walk through one part of the lab where you had to figure something out on your own and explain how you got there.
3. How did you verify that your work was correct? What would you look for to catch a mistake, whether it came from AI or from your own reasoning?
4. What is one thing you would do differently next time, either in how you approached the lab or in how you used (or did not use) AI?


Record your findings in the cell below.

1. I used AI tools during this lab to help with debugging and checking whether my RAG design choices made sense. I used ChatGPT when creating the RAG prompt template, assembling the LangChain pipeline, and interpreting retrieval results.

2. One useful prompt was asking, "why are my retrieved chunks off topic with a query about a specific product?" This helped me understand that retrieval problems are not always caused by chunk size and that query wording and retrieval settings can also affect results.

3. I verified my work by running each component separately before combining everything into the full RAG chain. For example, I inspected retrieved chunks and similarity scores to confirm that the retriever was finding relevant information, and I tested different customer questions to see whether the generated answers matched the knowledge base. 

4. One thing I would do differently next time is spend more time testing different retrieval strategies before changing the system.